# 01 — Data Loading, Cleaning & Data Quality Validation
### Healthcare Claims Reference Data Quality & Analytics Platform

**Business context:** This notebook ingests the raw synthetic claims ecosystem
(patients, providers, insurance plans, ICD/CPT reference codes, and claims),
performs technical data cleaning, applies the healthcare-specific data quality
(DQ) rules, and produces two outputs for downstream use:

1. `data/cleaned/` — load-ready CSVs (technically clean, correctly typed) that will later be loaded into MySQL
2. `outputs/data_quality_report.csv` — a DQ report summarizing exactly which claims have issues and why, the kind of report a claims operations team would review daily

**Design principle — cleaning vs. validation are different jobs:**
- *Cleaning* = technical fixes (correct dtypes, trim whitespace, remove exact duplicate rows caused by file re-ingestion). Cleaning never hides a business problem.
- *Validation* = business rule checks (missing/invalid codes, invalid provider, duplicate claim submissions, future dates, negative amounts, invalid status, missing plan). These are **flagged and reported**, not silently deleted — a real claims team needs to see and act on them.


## Step 1 — Imports & configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import date

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RAW_DIR = '../data/raw'
CLEAN_DIR = '../data/cleaned'
OUT_DIR = '../outputs'

print("Environment ready.")

Environment ready.


## Step 2 — Load raw data with explicit dtypes

**Why explicit dtypes matter here:** several ID/code columns (`cpt_code`, `provider_id`,
`plan_id`, `npi`) look numeric but are really categorical identifiers with meaningful
leading zeros (e.g. CPT `00000`). If we let pandas auto-infer types, it will silently
convert these to integers and strip the leading zeros — corrupting the codes before
we've even started. We force everything identifier-like to `str` up front.

In [2]:
dtype_claims = {
    'claim_id': str, 'patient_id': str, 'provider_id': str, 'plan_id': str,
    'icd_code': str, 'cpt_code': str, 'claim_status': str,
}

claims_raw = pd.read_csv(f'{RAW_DIR}/claims.csv', dtype=dtype_claims)
patients_raw = pd.read_csv(f'{RAW_DIR}/patients.csv', dtype=str)
providers_raw = pd.read_csv(f'{RAW_DIR}/providers.csv', dtype=str)
plans_raw = pd.read_csv(f'{RAW_DIR}/insurance_plans.csv', dtype=str)
icd_raw = pd.read_csv(f'{RAW_DIR}/icd_codes.csv', dtype=str)
cpt_raw = pd.read_csv(f'{RAW_DIR}/cpt_codes.csv', dtype=str)

# claim_amount is genuinely numeric (needed for aggregation), parse separately
claims_raw['claim_amount'] = pd.to_numeric(
    pd.read_csv(f'{RAW_DIR}/claims.csv')['claim_amount'], errors='coerce'
)

print(f"claims:           {claims_raw.shape}")
print(f"patients:         {patients_raw.shape}")
print(f"providers:        {providers_raw.shape}")
print(f"insurance_plans:  {plans_raw.shape}")
print(f"icd_codes:        {icd_raw.shape}")
print(f"cpt_codes:        {cpt_raw.shape}")

claims:           (10060, 9)
patients:         (2500, 4)
providers:        (300, 5)
insurance_plans:  (15, 3)
icd_codes:        (150, 2)
cpt_codes:        (113, 2)


In [3]:
claims_raw.head()

,claim_id,patient_id,provider_id,plan_id,icd_code,cpt_code,claim_date,claim_amount,claim_status
0,CLM0008295,PAT001387,PRV00132,PLN005,D69.6,93000,2024-01-12,498.65,Approved
1,CLM0003132,PAT000482,PRV00185,PLN008,R56.9,99211,2024-08-15,313.12,Approved
2,CLM0002778,PAT000530,PRV00191,PLN010,Z12.11,93880,2023-05-31,784.50,Pending
3,CLM0009306,PAT000104,PRV00016,PLN015,E66.9,99381,2023-05-04,249.02,Approved
4,CLM0000108,PAT000372,PRV00075,PLN014,R41.0,94060,2023-03-01,722.70,Approved


## Step 3 — Initial data profiling

Before touching anything, profile the raw claims table: nulls per column, dtypes,
and a first look at claim_status/claim_amount ranges. This is standard practice —
you always profile before you clean, so you can prove what you fixed.

In [4]:
profile = pd.DataFrame({
    'dtype': claims_raw.dtypes.astype(str),
    'null_count': claims_raw.isna().sum(),
    'null_pct': (claims_raw.isna().mean() * 100).round(2),
})
profile

,dtype,null_count,null_pct
claim_id,str,0,0.0
patient_id,str,0,0.0
provider_id,str,0,0.0
plan_id,str,60,0.6
icd_code,str,60,0.6
cpt_code,str,60,0.6
claim_date,str,0,0.0
claim_amount,float64,0,0.0
claim_status,str,0,0.0


In [5]:
print("claim_amount summary (raw, includes negative/dirty values):")
claims_raw['claim_amount'].describe()

claim_amount summary (raw, includes negative/dirty values):


count    10060.000000
mean       382.656024
std        214.780580
min      -1042.940000
25%        234.530000
50%        346.910000
75%        495.992500
max       1895.720000
Name: claim_amount, dtype: float64

## Step 4 — Technical cleaning

Cleaning steps applied (technical only, no business-rule judgment yet):

1. Trim leading/trailing whitespace from all string/ID columns
2. Standardize `claim_status` casing (Title Case)
3. Parse `claim_date` to an actual date type, keeping the original string for
   any row that fails to parse (so we don't lose information)
4. Drop exact full-row duplicates (identical across *every* column) — these are
   pure ingestion artifacts, not business duplicate claims (which we handle
   separately in validation, since they have *different* claim_ids)


In [6]:
def clean_claims(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply technical (non-business-rule) cleaning to the raw claims dataframe.

    Business purpose: standardizes formatting so downstream validation and
    MySQL loading behave predictably, WITHOUT making any judgment call about
    whether a claim is valid -- that is the validation step's job.
    """
    out = df.copy()

    # 1. Trim whitespace on all string columns
    str_cols = out.select_dtypes(include='object').columns
    for c in str_cols:
        out[c] = out[c].str.strip()

    # 2. Standardize claim_status casing
    out['claim_status'] = out['claim_status'].str.title()

    # 3. Parse claim_date safely
    out['claim_date_parsed'] = pd.to_datetime(out['claim_date'], errors='coerce')

    # 4. Drop exact full-row duplicates (ingestion artifacts)
    before = len(out)
    out = out.drop_duplicates()
    removed = before - len(out)

    print(f"Removed {removed} exact full-row duplicate rows.")
    return out


claims_clean_stage = clean_claims(claims_raw)
claims_clean_stage.head()

Removed 0 exact full-row duplicate rows.


/tmp/ipykernel_615/348004088.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = out.select_dtypes(include='object').columns


,claim_id,patient_id,provider_id,plan_id,icd_code,cpt_code,claim_date,claim_amount,claim_status,claim_date_parsed
0,CLM0008295,PAT001387,PRV00132,PLN005,D69.6,93000,2024-01-12,498.65,Approved,2024-01-12
1,CLM0003132,PAT000482,PRV00185,PLN008,R56.9,99211,2024-08-15,313.12,Approved,2024-08-15
2,CLM0002778,PAT000530,PRV00191,PLN010,Z12.11,93880,2023-05-31,784.50,Pending,2023-05-31
3,CLM0009306,PAT000104,PRV00016,PLN015,E66.9,99381,2023-05-04,249.02,Approved,2023-05-04
4,CLM0000108,PAT000372,PRV00075,PLN014,R41.0,94060,2023-03-01,722.70,Approved,2023-03-01


## Step 5 — Load reference tables (ICD, CPT, Providers, Plans) as lookup sets

These become the "source of truth" sets used to validate every claim.

In [7]:
valid_icd_codes = set(icd_raw['icd_code'])
valid_cpt_codes = set(cpt_raw['cpt_code'])
valid_provider_ids = set(providers_raw['provider_id'])
valid_plan_ids = set(plans_raw['plan_id'])
valid_patient_ids = set(patients_raw['patient_id'])
valid_statuses = {'Approved', 'Rejected', 'Pending'}

print(f"Reference sizes -> ICD: {len(valid_icd_codes)}, CPT: {len(valid_cpt_codes)}, "
      f"Providers: {len(valid_provider_ids)}, Plans: {len(valid_plan_ids)}, "
      f"Patients: {len(valid_patient_ids)}")

Reference sizes -> ICD: 150, CPT: 113, Providers: 300, Plans: 15, Patients: 2500


## Step 6 — Data quality validation rules

Each rule below implements one business validation requirement. Each returns a
boolean Series (`True` = row FAILS that rule) so we can flag, count, and report
on every issue independently — a claim can fail more than one rule at once.


In [8]:
def flag_missing_icd(df):
    """Flag claims with a missing (null) ICD diagnosis code."""
    return df['icd_code'].isna() | (df['icd_code'].str.len() == 0)


def flag_missing_cpt(df):
    """Flag claims with a missing (null) CPT procedure code."""
    return df['cpt_code'].isna() | (df['cpt_code'].str.len() == 0)


def flag_invalid_icd(df):
    """Flag claims whose ICD code is present but not in the reference table."""
    present = df['icd_code'].notna() & (df['icd_code'].str.len() > 0)
    return present & ~df['icd_code'].isin(valid_icd_codes)


def flag_invalid_cpt(df):
    """Flag claims whose CPT code is present but not in the reference table."""
    present = df['cpt_code'].notna() & (df['cpt_code'].str.len() > 0)
    return present & ~df['cpt_code'].isin(valid_cpt_codes)


def flag_invalid_provider(df):
    """Flag claims referencing a provider_id not found in the Providers table."""
    return ~df['provider_id'].isin(valid_provider_ids)


def flag_missing_plan(df):
    """Flag claims with a missing insurance plan_id."""
    return df['plan_id'].isna() | (df['plan_id'].str.len() == 0)


def flag_future_date(df):
    """Flag claims with a claim_date later than today (data entry error)."""
    today = pd.Timestamp(date.today())
    return df['claim_date_parsed'] > today


def flag_negative_amount(df):
    """Flag claims with a negative claim_amount (should never be negative)."""
    return df['claim_amount'] < 0


def flag_invalid_status(df):
    """Flag claims whose claim_status is outside the allowed status set."""
    return ~df['claim_status'].isin(valid_statuses)


def flag_duplicate_claims(df):
    """
    Flag claims that are duplicate SUBMISSIONS -- i.e. a different claim_id
    but identical business content (same patient, provider, plan, codes,
    date, and amount). This catches the common real-world case of a claim
    being accidentally submitted twice.
    """
    business_key = ['patient_id', 'provider_id', 'plan_id', 'icd_code',
                     'cpt_code', 'claim_date', 'claim_amount']
    return df.duplicated(subset=business_key, keep='first')


print("Validation rule functions defined.")

Validation rule functions defined.


## Step 7 — Apply all rules and build the flag matrix

Every claim gets one boolean column per rule, plus a rollup `is_valid` column
and an `issue_count` column. This is the core data structure the DQ report and
KPI notebook will both build on.

In [9]:
df = claims_clean_stage.copy()

df['flag_missing_icd']       = flag_missing_icd(df)
df['flag_missing_cpt']       = flag_missing_cpt(df)
df['flag_invalid_icd']       = flag_invalid_icd(df)
df['flag_invalid_cpt']       = flag_invalid_cpt(df)
df['flag_invalid_provider']  = flag_invalid_provider(df)
df['flag_missing_plan']      = flag_missing_plan(df)
df['flag_future_date']       = flag_future_date(df)
df['flag_negative_amount']   = flag_negative_amount(df)
df['flag_invalid_status']    = flag_invalid_status(df)
df['flag_duplicate_claim']   = flag_duplicate_claims(df)

flag_cols = [c for c in df.columns if c.startswith('flag_')]
df['issue_count'] = df[flag_cols].sum(axis=1)
df['is_valid'] = df['issue_count'] == 0

print(f"Total claims processed: {len(df):,}")
print(f"Valid claims:           {df['is_valid'].sum():,}")
print(f"Claims with >=1 issue:  {(~df['is_valid']).sum():,}")

Total claims processed: 10,060
Valid claims:           9,460
Claims with >=1 issue:  600


## Step 8 — Data Quality Report

One row per rule, with count, percentage of total claims, and example claim_ids — exactly what an operations team would want in a daily/weekly DQ summary.

In [10]:
def build_dq_report(df, flag_cols, n_examples=5):
    """
    Build a summary Data Quality Report: one row per validation rule with
    the count of affected claims, percentage of total, and example claim_ids
    so an analyst can immediately pull up real examples to investigate.
    """
    rows = []
    total = len(df)
    for col in flag_cols:
        issue_name = col.replace('flag_', '').replace('_', ' ').title()
        affected = df[df[col]]
        rows.append({
            'issue_type': issue_name,
            'affected_claims': int(affected.shape[0]),
            'pct_of_total': round(affected.shape[0] / total * 100, 2),
            'example_claim_ids': ', '.join(affected['claim_id'].head(n_examples).tolist()),
        })
    report = pd.DataFrame(rows).sort_values('affected_claims', ascending=False).reset_index(drop=True)
    return report


dq_report = build_dq_report(df, flag_cols)
dq_report

,issue_type,affected_claims,pct_of_total,example_claim_ids
0,Missing Icd,60,0.6,"CLM0006741, CLM0002639, CLM0000488, CLM0006066..."
1,Missing Cpt,60,0.6,"CLM0000036, CLM0004217, CLM0004197, CLM0006337..."
2,Invalid Icd,60,0.6,"CLM0009674, CLM0006858, CLM0003316, CLM0003028..."
3,Invalid Cpt,60,0.6,"CLM0003208, CLM0002110, CLM0004116, CLM0002575..."
4,Invalid Provider,60,0.6,"CLM0001319, CLM0009243, CLM0009025, CLM0003542..."
5,Missing Plan,60,0.6,"CLM0001057, CLM0007442, CLM0009933, CLM0003413..."
6,Future Date,60,0.6,"CLM0009969, CLM0008758, CLM0005573, CLM0009061..."
7,Negative Amount,60,0.6,"CLM0004996, CLM0001225, CLM0005690, CLM0000350..."
8,Invalid Status,60,0.6,"CLM0006750, CLM0002689, CLM0000040, CLM0007648..."
9,Duplicate Claim,60,0.6,"CLM0010050, CLM0010002, CLM0008804, CLM0010038..."


## Step 9 — Export cleaned datasets and reports

- `claims_valid.csv` — clean claims only (ready for MySQL fact table load / trusted reporting)
- `claims_flagged.csv` — full claim set with all flag columns, for an ops review queue
- `data_quality_report.csv` — the summary report above
- Reference tables copied through unchanged (already clean)

In [11]:
import os
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

export_cols = ['claim_id', 'patient_id', 'provider_id', 'plan_id', 'icd_code',
               'cpt_code', 'claim_date', 'claim_amount', 'claim_status']

claims_valid = df.loc[df['is_valid'], export_cols].reset_index(drop=True)
claims_flagged = df.drop(columns=['claim_date_parsed']).reset_index(drop=True)

claims_valid.to_csv(f'{CLEAN_DIR}/claims_valid.csv', index=False)
claims_flagged.to_csv(f'{CLEAN_DIR}/claims_flagged.csv', index=False)
dq_report.to_csv(f'{OUT_DIR}/data_quality_report.csv', index=False)

patients_raw.to_csv(f'{CLEAN_DIR}/patients.csv', index=False)
providers_raw.to_csv(f'{CLEAN_DIR}/providers.csv', index=False)
plans_raw.to_csv(f'{CLEAN_DIR}/insurance_plans.csv', index=False)
icd_raw.to_csv(f'{CLEAN_DIR}/icd_codes.csv', index=False)
cpt_raw.to_csv(f'{CLEAN_DIR}/cpt_codes.csv', index=False)

print(f"claims_valid.csv        -> {len(claims_valid):,} rows (load-ready)")
print(f"claims_flagged.csv      -> {len(claims_flagged):,} rows (full set + flags)")
print(f"data_quality_report.csv -> {len(dq_report)} rule rows")
print("Reference tables copied to data/cleaned/.")

claims_valid.csv        -> 9,460 rows (load-ready)
claims_flagged.csv      -> 10,060 rows (full set + flags)
data_quality_report.csv -> 10 rule rows
Reference tables copied to data/cleaned/.


## Summary

- Started with **10,060** raw claim records
- Removed exact full-row duplicates during technical cleaning
- Applied **10 business data-quality rules**
- Flagged claims are **kept, not deleted** — they're exported separately for an operations review queue
- Only fully valid claims move forward into `claims_valid.csv`, which is what the MySQL fact table and business KPIs will be built from

**Next step:** Notebook `02_kpi_business_reporting.ipynb` — calculate the full business KPI suite (approval/rejection rates, claims by month/provider/plan/state, top diagnosis & procedure codes) from `claims_valid.csv`.